In [55]:
!pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 1.8 MB/s eta 0:00:00a 0:00:01


In [69]:
!pip install ftfy

In [65]:
import random
import re
import json
from itertools import islice
import ftfy
from tqdm import tqdm
from collections import defaultdict
import hashlib
from datasketch import MinHash, MinHashLSH
import numpy as np

In [6]:
is_empty = lambda text: text is None or text.strip() == ""
def write_jsonl(records, path):
    with open(path, "w", encoding="UTF-8") as out:
        c = 0
        for obj in records:
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
            c += 1
    return c
def read_jsonl(path):
    with open(path, encoding="UTF-8") as data:
        for obj in data:
            if is_empty(obj):
                continue
            yield json.loads(obj)

In [7]:
pretrain = "/kaggle/input/datasets/absolute1000/minillm-corpus-raw/PRETRAIN.jsonl"
sft = "/kaggle/input/datasets/absolute1000/minillm-corpus-raw/SFT.jsonl"

c1 = 0
for i in read_jsonl(pretrain):
    c1 += 1
c2 = 0
for i in read_jsonl(sft):
    c2 += 1
print(c1, c2)

2182000 46669


In [24]:
def normalize_text(text):
    text = ftfy.fix_text(text)
    text = text.replace("\u00a0", " ")
    patterns = {
        "spaces_at_the_end_of_each_line": [re.compile(r"[ \t]+$", re.MULTILINE), ""],
        "repeated_spaces_within_a_line_to_a_single_space": [re.compile(r"(?<=\S) {2,}"), " "],
    }
    for rule in patterns:
        pattern = patterns[rule]
        text = pattern[0].sub(pattern[1], text)
    text = text.strip()
    return text
def normalize_record(record):
    if record.get("turns", None) is None:
        record["text"] = normalize_text(record["text"])
    else:
        for phrase in record["turns"]:
            phrase["text"] = normalize_text(phrase["text"])
    return record
            

In [26]:
sft_normalized = "/kaggle/working/sft_normalized.jsonl"
temp = []
for raw in read_jsonl(sft):
    temp.append(normalize_record(raw))
write_jsonl(temp, sft_normalized)

46669

In [27]:
def filter1sft(text):
    return is_empty(text)
def filter2sft(text, param=1000):
    return len(text) > param
def filter3sft(text):
    return re.search(r"\bLl\b", text)

def filter1pretrain(text, param=200):
    return len(text) < param
def filter2pretrain(text):
    return re.search(r"[a-z]\?[a-z]", text)
def filter3pretrain(record, th=2.5):
    return record["meta"]["score"] < th

In [34]:
def count_filters_sft(path):
    csft = defaultdict(lambda: defaultdict(int))
    catch_bugs_Ll = []
    for raw in read_jsonl(path):
        ispure = True
        if any(filter1sft(t["text"]) for t in raw["turns"]):
            csft["filter1sft"][raw["source"]] += 1
            ispure = False
        if any(filter2sft(t["text"]) for t in raw["turns"]):
            csft["filter2sft"][raw["source"]] += 1
            ispure = False
        hit = next((t["text"] for t in raw["turns"] if filter3sft(t["text"])), None)
        if hit is not None:
            csft["filter3sft"][raw["source"]] += 1            
            ispure = False
            if len(catch_bugs_Ll) <= 20:
                catch_bugs_Ll.append(hit)
        if ispure:
            csft["pure"][raw["source"]] += 1
    return csft, catch_bugs_Ll
def count_filters_pretrain(path, par1=200, par3=2.5):
    cpretrain = defaultdict(lambda: defaultdict(int))
    catch_bugs = []
    for raw in islice(read_jsonl(path), 100000):
        ispure = True
        raw = normalize_record(raw)
        if filter1pretrain(raw["text"], par1):
            ispure = False
            cpretrain["filter1pretrain"][raw["source"]] += 1
        if temp:= filter2pretrain(raw["text"]):
            ispure = False
            cpretrain["filter2pretrain"][raw["source"]] += 1
            if len(catch_bugs) < 20:
                catch_bugs.append(raw["text"][max(0, temp.start() - 40):temp.start() + 40])
        if filter3pretrain(raw, par3):
            ispure = False
            cpretrain["filter3pretrain"][raw["source"]] += 1
        if ispure:
            cpretrain["pure"][raw["source"]] += 1
    return cpretrain, catch_bugs

In [35]:
csft, catch_bugs_Ll = count_filters_sft(sft_normalized)
print({k: dict(v) for k, v in csft.items()})
print(catch_bugs_Ll)

{'pure': {'mbpp': 427, 'dailydialog': 13112, 'empatheticdialogues': 23041, 'personachat': 8422, 'llmdistillated': 1308}, 'filter2sft': {'dailydialog': 5}, 'filter3sft': {'dailydialog': 1, 'personachat': 353}}
['To which gate do I need to go to catch connection flight Ll 01 to Madrid?', 'Ll. I love black too! Guess I was playing my music too loud.', "Ll well that's interesting. You like dogs? Pugs are so loyal, I have one name biscuit.", "Ll I've 3, my parents do not know.", 'Ll. sounds fun, but be careful with that. Gold coins are heavy!', 'Ll.', 'Ll, not exactly. Comic book artist and cartoonist are very well off.', 'Ll that mixed up would taste weird I like salads. Was nice talking to you.', 'Ll. Bubble gum. I love me some steak.', 'You can barely tell the numbers on it. Ll.', 'He can jump a 6 foot fence from a standing position. Ll.', "Ll there are a lot of misconceptions about them but they're definitely real.", 'Dude, you need therapy! Ll.', "Ll that's whats up! I smoke all day an

In [36]:
cpretrain, catch_bugs = count_filters_pretrain(pretrain)
print({k: dict(v) for k, v in cpretrain.items()})
print(catch_bugs)

{'pure': {'fineweb-edu': 99075}, 'filter2pretrain': {'fineweb-edu': 925}}
['improvise on their own; playtime versus r?m?uilding.\nDiscussions about play forc', 'dc.gov/mmwr/preview/mmwrhtml/ss6103a1.htm?s_cid=ss6103a1_w\nTo the Community Repo', 'y.si.edu/collections/surveying/object.cfm?recordnumber=747113; Peggy A. Kidwell,', 'L:http://www.healthscout.com/template.asp?id=669525', 'tness: http://aawsat.com/english/news.asp?section=2&id=19640\nMemories of a high ', 'ver/DocumentSet/Zoon_report_2006_en,0.pdf?ssbinary=true\n- Corbel MJ. Brucellosis', 'ient. Granite is formed when liquid magma?molten rock material?cools beneath the', ' http://wiki.crin.org/mediawiki/index.php?title=Main_Page\nWe are launching the W', 'tp://www.genealogical.com/item_detail.asp?afid=&ID=4887\nMagna Carta Ancestry: A ', 'w.eatrightontario.ca/en/viewdocument.aspx?id=39. Accessed June 30, 2011.\n- Duyff', "ine, 'Hey, what would Aristotle have said?what about this?' And maybe we won't g", '.abc-clio.com.ilsweb.l

In [38]:
c = 0
temp = []
sftfiltered = "/kaggle/working/sft_filtered.jsonl"
for record in read_jsonl(sft_normalized):
    if record["source"] == "personachat":
        c_ = 0
        for phrase in record["turns"]:
            new = re.sub(r"\bLl\b", "lol", phrase["text"])
            if new != phrase["text"]:
                c_ += 1
                phrase["text"] = new
        if c_ != 0:
            c += 1
    temp.append(record)
print(c)
write_jsonl(temp, sftfiltered)

353


46669

In [43]:
def record_text(record):
    if record.get("turns", None) is None:
        return record["text"]
    else:
        return '\n'.join([i["text"] for i in record["turns"]])

In [47]:
def exact_dedup(records):
    c = 0
    s = set()
    for raw in records:
        text = record_text(raw)
        encoded = hashlib.sha256(text.encode()).hexdigest()
        if encoded in s:
            c += 1
        else:
            s.add(encoded)
            yield raw
    print(c)      

In [49]:
pretrainnormalized = "/kaggle/working/pretrain_normalized.jsonl"
write_jsonl(exact_dedup(map(normalize_record, read_jsonl(pretrain))), pretrainnormalized)

18455


2163545

In [50]:
sftsoftdedup = "/kaggle/working/sft_soft_dedup.jsonl"
write_jsonl(exact_dedup(map(normalize_record, read_jsonl(sftfiltered))), sftsoftdedup)

886


45783

In [52]:
def near_dedup(records, dup_counts, dup_examples, threshold=0.8):
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    texts = dict()
    for raw in records:
        text_ = record_text(raw)
        text = record_text(raw).lower().split()
        m = MinHash(num_perm=128)
        if len(text) >= 3:
            for i in range(len(text) - 2):
                t = ' '.join(text[i:i+3])
                m.update(t.encode("utf8"))
        else:
            m.update(text_.lower().encode("utf8"))

        key = "|".join([raw["source"], raw["source_split"], raw["source_id"]])
        res = lsh.query(m)
        if len(res) == 0:
            lsh.insert(key, m)
            texts[key] = text_.lower()
            yield raw
        else:
            dup_counts[raw["source"]] = dup_counts.get(raw["source"], 0) + 1
            if len(dup_examples) < 10:
                dup_examples.append([texts[res[0]], text_.lower()])

In [57]:
sftharddedup = "/kaggle/working/sft_hard_dedup.jsonl"
dup_counts = {}
dup_examples = []
write_jsonl(near_dedup(read_jsonl(sftsoftdedup), dup_counts, dup_examples), sftharddedup)
print(dup_counts)
print(dup_examples)

308004it [2:39:42, 32.14it/s]


{'dailydialog': 651, 'llmdistillated': 4}
[["hi, my name is lean, and i'm from russia.\nnice to meet you, lean. my name is alike. i'm from japan.\nto me english is a difficult language.\na second language is always difficult.\ntrue, but english is harder than most. it's a crazy language.\na crazy language? why do you say that?\none letter can have several pronunciations and one word can have several meanings.\nno wonder you say english is a crazy language.", "hi, my name is lean, and i'm from russia.\nnice to meet you, lvan. my name is alike. i'm from japan.\nto me english is a difficult language.\na second language is always difficult.\ntrue, but english is harder than most. it's a crazy language.\na crazy language? why do you say that?\none letter can have several pronunciations and one word can have several meanings.\nno wonder you say english is a crazy language."], ["what ' s up? you don ' t look too good.\nyeah, my head hurts, that ' s all. i ' ve been in physics class all day. i

In [59]:
finalsft = "/kaggle/working/sft_final.jsonl"
fix_counts = {"apostrophe": 0, "space_before_punct": 0}

def fix_dailydialog(record):
    if record["source"] != "dailydialog":
        return record
    for phrase in record["turns"]:
        text = phrase["text"]
        new = re.sub(r"(?<=\w) ' (?=\w)", "'", text)
        if new != text:
            fix_counts["apostrophe"] += 1
        newer = re.sub(r" +(?=[,.?!;:])", "", new)
        if newer != new:
            fix_counts["space_before_punct"] += 1
        phrase["text"] = newer
    return record

print(write_jsonl(map(fix_dailydialog, read_jsonl(sftharddedup)), finalsft))
print(fix_counts)

45128
{'apostrophe': 8122, 'space_before_punct': 0}


In [61]:
def split_of(record, val_share, test_share):
    val_share = val_share / 100
    test_share = test_share / 100
    string = "|".join([raw["source"], raw["source_split"], raw["source_id"]])
    hash_str = hashlib.sha256(string.encode()).hexdigest()
    hash_int = int(hash_str, 16) % 1000
    test_tr = 1000 - 1000 * test_share
    train_tr = test_tr - 1000 * val_share
    if hash_int < train_tr:
        ans = "train"
    elif hash_int < test_tr:
        ans = "valid"
    else:
        ans = "test"
    return ans

In [62]:
TRAIN_SFT = "/kaggle/working/TRAIN_SFT.jsonl"
VALID_SFT = "/kaggle/working/VALID_SFT.jsonl"
TEST_SFT = "/kaggle/working/TEST_SFT.jsonl"

with (open(TRAIN_SFT, "w", encoding="UTF-8") as train,
     open(VALID_SFT, "w", encoding="UTF-8") as val,
     open(TEST_SFT, "w", encoding="UTF-8") as test):
         for raw in read_jsonl(finalsft):
             ans = split_of(raw, 1.5, 1.5)
             if ans == "train":
                 train.write(json.dumps(raw, ensure_ascii=False) + "\n")
             elif ans == "valid":
                 val.write(json.dumps(raw, ensure_ascii=False) + "\n")
             else:
                 test.write(json.dumps(raw, ensure_ascii=False) + "\n")

In [63]:
!wc -l /kaggle/working/*_SFT.jsonl

     704 /kaggle/working/TEST_SFT.jsonl
   43732 /kaggle/working/TRAIN_SFT.jsonl
     692 /kaggle/working/VALID_SFT.jsonl
   45128 total


In [66]:
PERCENTILES = (50, 75, 90, 95, 99)
CHARS_PER_TOKEN = 3.5

def pct(values):
    return {p: int(np.percentile(values, p)) for p in PERCENTILES}

def show(name, values):
    chars = pct(values)
    tokens = {p: int(v / CHARS_PER_TOKEN) for p, v in chars.items()}
    print(f"{name}  (n={len(values)})")
    print("  chars: ", chars)
    print("  ~tokens:", tokens)

turn_all, turn_bot, dialog_len, n_turns = [], [], [], []
dialog_by_source = defaultdict(list)
for rec in read_jsonl(TRAIN_SFT):
    for t in rec["turns"]:
        turn_all.append(len(t["text"]))
        if t["role"] == "bot":
            turn_bot.append(len(t["text"]))
    length = len(record_text(rec))
    dialog_len.append(length)
    n_turns.append(len(rec["turns"]))
    dialog_by_source[rec["source"]].append(length)

print("=" * 100)
print("SFT")
show("reply (any role)", turn_all)
show("reply (bot only)", turn_bot)
show("whole dialog", dialog_len)
print("turns per dialog:", pct(n_turns))
print()
for source, values in dialog_by_source.items():
    show(f"whole dialog | {source}", values)

doc_len = [len(rec["text"]) for rec in itertools.islice(read_jsonl(pretrainnormalized), 100000)]
print("=" * 100)
print("PRETRAIN (first 100k docs)")
show("document", doc_len)

SFT
reply (any role)  (n=314285)
  chars:  {50: 50, 75: 69, 90: 96, 95: 124, 99: 199}
  ~tokens: {50: 14, 75: 19, 90: 27, 95: 35, 99: 56}
reply (bot only)  (n=153100)
  chars:  {50: 48, 75: 67, 90: 88, 95: 115, 99: 193}
  ~tokens: {50: 13, 75: 19, 90: 25, 95: 32, 99: 55}
whole dialog  (n=43732)
  chars:  {50: 327, 75: 578, 90: 790, 95: 892, 99: 1196}
  ~tokens: {50: 93, 75: 165, 90: 225, 95: 254, 99: 341}
turns per dialog: {50: 5, 75: 11, 90: 16, 95: 16, 99: 16}

whole dialog | mbpp  (n=416)
  chars:  {50: 229, 75: 289, 90: 404, 95: 460, 99: 676}
  ~tokens: {50: 65, 75: 82, 90: 115, 95: 131, 99: 193}
whole dialog | dailydialog  (n=11197)
  chars:  {50: 364, 75: 590, 90: 904, 95: 1104, 99: 1547}
  ~tokens: {50: 104, 75: 168, 90: 258, 95: 315, 99: 442}
whole dialog | empatheticdialogues  (n=22327)
  chars:  {50: 270, 75: 352, 90: 454, 95: 542, 99: 750}
  ~tokens: {50: 77, 75: 100, 90: 129, 95: 154, 99: 214}
whole dialog | personachat  (n=8516)
  chars:  {50: 721, 75: 815, 90: 901, 95: 95

In [67]:
random.seed(42)
GATE_FILE = "/kaggle/working/gate_examples.txt"
PRETRAIN_COUNT = 2163545

sft_records = list(read_jsonl(TRAIN_SFT))
sft_sample = random.sample(sft_records, 100)

pretrain_idx = set(random.sample(range(PRETRAIN_COUNT), 100))
pretrain_sample = []
for i, rec in enumerate(tqdm(read_jsonl(pretrainnormalized))):
    if i in pretrain_idx:
        pretrain_sample.append(rec)

with open(GATE_FILE, "w", encoding="UTF-8") as out:
    out.write("=" * 40 + " SFT " + "=" * 40 + "\n\n")
    for n, rec in enumerate(sft_sample, 1):
        out.write(f"### SFT {n} | {rec['source']} | {rec['source_split']} | {rec['source_id']}\n")
        for t in rec["turns"]:
            out.write(f"{t['role']}: {t['text']}\n")
        out.write("\n")

    out.write("=" * 40 + " PRETRAIN " + "=" * 40 + "\n\n")
    for n, rec in enumerate(pretrain_sample, 1):
        out.write(f"### PRETRAIN {n} | {rec['source_id']}\n")
        out.write(rec["text"][:500] + "\n\n")

print(len(sft_sample), len(pretrain_sample))

2163545it [01:48, 19895.36it/s]

100 100


In [68]:
import os

TO_DELETE = [
    "/kaggle/working/sft_normalized.jsonl",
    "/kaggle/working/sft_filtered.jsonl",
    "/kaggle/working/sft_soft_dedup.jsonl",
    "/kaggle/working/sft_hard_dedup.jsonl",
]
for path in TO_DELETE:
    if os.path.exists(path):
        os.remove(path)
        print("deleted:", path)
    else:
        print("not found:", path)

!ls -lh /kaggle/working/
!df -h /kaggle/working

deleted: /kaggle/working/sft_normalized.jsonl
deleted: /kaggle/working/sft_filtered.jsonl
deleted: /kaggle/working/sft_soft_dedup.jsonl
deleted: /kaggle/working/sft_hard_dedup.jsonl
total 11G
-rw-r--r-- 1 root root 104K Sep 30 00:48 gate_examples.txt
-rw-r--r-- 1 root root  11G Sep 30 00:16 pretrain_normalized.jsonl
-rw-r--r-- 1 root root  39M Sep 30 00:35 sft_final.jsonl
-rw-r--r-- 1 root root 618K Sep 30 00:39 TEST_SFT.jsonl
-rw-r--r-- 1 root root  38M Sep 30 00:39 TRAIN_SFT.jsonl
-rw-r--r-- 1 root root 598K Sep 30 00:39 VALID_SFT.jsonl
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   11G  9.2G  53% /kaggle/working
